# MLflow Master Notebook — Core MLOps Reference

This notebook is a **study/reference notebook**, not a short crash course.

The structure is intentionally:

> **One Markdown explanation → one Code cell → repeat**

The explanations are written in Arabic, while MLflow API names and technical terms stay in English.

## Sources

### Primary source
- `mlflow_crash_course.ipynb` — the notebook supplied with this study.

### Official MLflow documentation
- Tracking: https://mlflow.org/docs/latest/ml/tracking
- Architecture: https://mlflow.org/docs/latest/self-hosting/architecture/overview/
- Backend Store: https://mlflow.org/docs/latest/self-hosting/architecture/backend-store/
- Artifact Store: https://mlflow.org/docs/latest/self-hosting/architecture/artifact-store/
- Tracking Server: https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/
- Model Registry: https://mlflow.org/docs/latest/ml/model-registry/
- Model Registry Workflow: https://mlflow.org/docs/latest/ml/model-registry/workflow
- Model Signatures: https://mlflow.org/docs/latest/ml/model/signatures/
- Python API Reference: https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html

## Important

The supplied crash-course notebook contains legacy Model Registry **Stages** such as `Staging` and `Production`.
MLflow's current documentation says Model Stages are deprecated. This notebook therefore teaches the **modern alias-based workflow** instead, while preserving the original notebook's concepts.

## 0. The Mental Model — What MLflow Is Actually Doing

Before memorizing functions, understand the objects and their relationships:

```text
ML Project
    │
    ▼
Experiment
    │
    ├── Run 1
    │     ├── Params
    │     ├── Metrics
    │     ├── Tags
    │     ├── Artifacts
    │     └── Model
    │
    ├── Run 2
    └── Run 3

Run / Model
    │
    ▼
Model Registry
    │
    ├── Registered Model
    │      ├── Version 1
    │      ├── Version 2
    │      └── Version 3
    │
    └── Alias: champion → one version
```

### Key definitions

- **Experiment**: A container that organizes A group of related runs   .
- **Run**: One execution of a training/evaluation process.
- **Parameter**: A configuration or input that affects the experiment.
- **Metric**: A value that measures a result or performance.
- **Tag**: Metadata used for classification and organization.
- **Artifact**: A file produced by the run  plot  JSON  model files.
- **MLflow Model**: model   MLflow   .
- **Registered Model**:     Model Registry.
- **Model Version**: A specific version  Registered Model.
- **Alias**: A mutable named reference   version  `champion`.

Core idea: MLflow     .        .

In [1]:
# Cell 0 — inspect the MLflow installation and core modules

import sys
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)
print("Python version:", sys.version.split()[0])
print("Tracking URI:", mlflow.get_tracking_uri())
print("MLflowClient:", MlflowClient)

C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow version: 3.10.1
Python version: 3.11.9
Tracking URI: sqlite:///C:/Users/PC/Downloads/mlflow.db
MLflowClient: <class 'mlflow.tracking.client.MlflowClient'>


## 1. Tracking URI — Where does MLflow store and retrieve tracking data?

`mlflow.set_tracking_uri(...)`      MLflow / tracking metadata.

   :

```python
mlflow.set_tracking_uri("sqlite:///mlflow_test.db")
```

   SQLite  **Backend Store**.

The Backend Store stores metadata :

- run ID
- experiment information
- parameters
- metrics
- tags
- timestamps
- model/trace metadata

Large files  model weights  are stored in **Artifact Store**.

     SQLite. in a team environment   Tracking Server  database  artifact storage .

In [2]:
# Cell 1 — configure one local database for this entire notebook

from pathlib import Path
import mlflow

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / "mlflow_master.db"

TRACKING_URI = f"sqlite:///{DB_PATH}"
mlflow.set_tracking_uri(TRACKING_URI)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Database path:", DB_PATH.resolve())

Tracking URI: sqlite:///c:\Users\PC\Downloads\mlflow_master.db
Database path: C:\Users\PC\Downloads\mlflow_master.db


## 2. Experiment — How do we organize  Runs

`Experiment`      Runs.

     Runs  :

```text
Run A
Run B
Run C
...
```

 :

```text
customer_churn
    ├── run A
    ├── run B
    └── run C
```

 API :

```python
mlflow.set_experiment("experiment_name")
```

    experiment  MLflow      .

   Experiment Tags  .

In [5]:
# Cell 2 — create/select an experiment and add experiment-level metadata

EXPERIMENT_NAME = "mlflow_master_experiment"

experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

mlflow.set_experiment_tag("project", "mlflow-master-notebook")
mlflow.set_experiment_tag("purpose", "learning-and-reference")

print("Experiment configured.")

Experiment name: mlflow_master_experiment
Experiment ID: 1
Experiment configured.


## 3. Run lifecycle —  object  Tracking

 Run   .

   Python :

```python
with mlflow.start_run() as run:
    ...
```

  `with`   run     .

  run :

```text
Params
Metrics
Tags
Artifacts
Model
```

   MLflow Tracking.

  :

```python
run.info.run_id
```

  Run ID          .

In [12]:
# Cell 3 — create the smallest useful MLflow run

with mlflow.start_run(run_name="first_reference_run") as run:
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_metric("accuracy", 0.91)
    mlflow.set_tag("purpose", "learning")

    run_id = run.info.run_id
    experiment_id = run.info.experiment_id

print("Run ID:", run_id)
print("Experiment ID:", experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.end_run()

Run ID: 89310a9f8c3749d59cd8ce7fb50d6571
Experiment ID: 1
Tracking URI: sqlite:///c:\Users\PC\Downloads\mlflow_master.db


## 4. Parameters — How was the experiment configured?

Parameter  configuration/input .

:

```text
learning_rate = 0.01
batch_size = 32
n_estimators = 100
max_depth = 10
optimizer = adam
```

    Parameter:

> **What did I configure before/during training?**

  parameter :

```python
mlflow.log_param("learning_rate", 0.01)
```

 :

```python
mlflow.log_params({...})
```

  Parameters  performance.   Metrics.

In [7]:
# Cell 4 — log single and multiple parameters

with mlflow.start_run(run_name="parameters_demo"):
    mlflow.log_param("algorithm", "random_forest")
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
    })

    print("Logged parameters.")

Logged parameters.


## 5. Metrics — What was the result of the experiment?

Metric    performance.

:

```text
accuracy
precision
recall
f1
rmse
mae
r2
loss
```

 :

```text
Parameter → configuration
Metric    → measurement
```

  metric   .

   metric  iterations/epochs  `step`   MLflow      .

In [8]:
# Cell 5 — log final metrics and a metric history

with mlflow.start_run(run_name="metrics_demo"):
    # One final value
    mlflow.log_metric("final_score", 0.95)

    # A time/step series
    for epoch in range(5):
        accuracy = 0.80 + epoch * 0.04
        loss = 1.0 / (epoch + 1)

        mlflow.log_metric("accuracy", accuracy, step=epoch)
        mlflow.log_metric("loss", loss, step=epoch)

    print("Logged final metrics and metric history.")

Logged final metrics and metric history.


## 6. Tags — metadata not performance

Tags  key/value metadata.

    :

```python
mlflow.set_tag("model_type", "random_forest")
mlflow.set_tag("dataset", "iris")
mlflow.set_tag("author", "MLflow Course Student")
```

 Tags  :

```text
environment = development
team = data-science
dataset_version = v3
model_type = random_forest
purpose = baseline
```

 :

- Parameter:   configuration.
- Metric:  performance.
- Tag:  /.

In [ ]:
# Cell 6 — run-level tags

with mlflow.start_run(run_name="tags_demo"):
    mlflow.log_params({
        "model": "random_forest",
        "n_estimators": 100,
    })

    mlflow.log_metric("rmse", 42.5)

    mlflow.set_tags({
        "environment": "development",
        "team": "data-science",
        "dataset_version": "v1",
        "run_type": "baseline",
    })

    print("Logged run-level tags.")

## 7. Artifacts —

Artifacts  files produced by a run.

:

- PNG plots
- JSON model cards
- CSV reports
- text summaries
- trained model files
- evaluation outputs

The Backend Store       . MLflow  metadata  artifacts.

 APIs :

```python
mlflow.log_artifact("file.txt")
mlflow.log_artifacts("directory/")
mlflow.log_dict({...}, "report.json")
mlflow.log_figure(fig, "plot.png")
```

In [ ]:
# Cell 7 — log files and structured artifacts

import json
from pathlib import Path

artifact_dir = Path("mlflow_demo_artifacts")
artifact_dir.mkdir(exist_ok=True)

text_file = artifact_dir / "notes.txt"
text_file.write_text("This file was produced during an MLflow run.\n", encoding="utf-8")

report = {
    "purpose": "MLflow artifact demo",
    "status": "complete",
    "important": True,
}

with mlflow.start_run(run_name="artifacts_demo"):
    mlflow.log_artifact(str(text_file), artifact_path="files")
    mlflow.log_dict(report, "reports/report.json")

    print("Artifacts logged.")

Artifacts logged.


## 8. Logging figures

Plots are artifacts, not metrics.

:

```text
RMSE = 42.1       → Metric
rmse_curve.png    → Artifact
```

   Matplotlib Seaborn  plots  training loss confusion matrix.

  `mlflow.log_figure()`   Matplotlib figure.

In [14]:
# Cell 8 — log a Matplotlib figure

import matplotlib.pyplot as plt

with mlflow.start_run(run_name="figure_demo"):
    steps = list(range(10))
    loss = [1 / (i + 1) for i in steps]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, loss)
    ax.set_title("Training Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True)

    mlflow.log_figure(fig, "plots/training_loss.png")
    plt.close(fig)

print("Figure logged as an artifact.")

Figure logged as an artifact.


## 9. Real ML experiment —

 crash-course  Diabetes dataset :

- LinearRegression
- RandomForestRegressor
- XGBoost

 model :

```text
train
  ↓
predict
  ↓
calculate metrics
  ↓
log parameters
  ↓
log metrics
  ↓
log model
```

     MLflow:

     model      Run     .

In [ ]:
# Cell 9 — compare multiple real ML models

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    random_state=42,
)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=50,random_state=42,),
}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
        })

        mlflow.set_tag("dataset", "sklearn_diabetes")
        mlflow.set_tag("model_name", model_name)

        print(f"{model_name:18s} RMSE={rmse:.3f}, R2={r2:.3f}")

LinearRegression   RMSE=53.853, R2=0.453
RandomForest       RMSE=55.174, R2=0.425


## 10. Model Logging —   Artifact  MLflow Model

  model    MLflow Model  structure metadata flavor information    MLflow.

  :

```python
mlflow.sklearn.log_model(...)
```

     pickle   MLOps workflow.

:

```text
trained sklearn model
        ↓
mlflow.sklearn.log_model()
        ↓
MLflow Model artifact
        ↓
load_model()
```

In [16]:
# Cell 10 — log a scikit-learn model

from mlflow.models import infer_signature

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

with mlflow.start_run(run_name="model_logging_demo") as run:
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    signature = infer_signature(X_test, predictions)

    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
    )

    mlflow.log_metric(
        "rmse",
        float(np.sqrt(mean_squared_error(y_test, predictions))),
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

2026/08/31 17:27:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 95ae66b5be2d4441a64b16c317de1f66
Model URI: models:/m-576482a165c9463881ec1e1340bfcedf


## 11. Model Signature —    model

`infer_signature(input, output)`  schema  model.

 signature :

```text
Input schema
Output schema
```

    deployment  application        model.

  :

```python
signature = infer_signature(X_test, y_pred)
```

  signature  `log_model()`.

  pattern     MLflow .

In [ ]:
# Cell 11 — inspect the signature of a logged model

from mlflow.models import infer_signature

with mlflow.start_run(run_name="signature_demo") as run:
    model = LinearRegression()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    signature = infer_signature(X_test, predictions)

    info = mlflow.sklearn.log_model(
        model,
        name="signature_model",
        signature=signature,
    )

    print("Model URI:", info.model_uri)
    print("Signature:")
    print(signature)

## 12. `search_runs()` — Searching for the best experiments

This is one of the most important APIs  MLflow.

  :

```python
mlflow.search_runs(...)
```

   :

```python
filter_string="metrics.accuracy > 0.7"
order_by=["metrics.accuracy DESC"]
```

:

```text
Experiment
   ↓
many Runs
   ↓
filter
   ↓
sort
   ↓
best candidates
```

    model-selection logic   model   UI.

In [17]:
# Cell 12 — search and rank runs

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.final_score DESC"],
)

columns = [
    c for c in [
        "run_id",
        "run_name",
        "status",
        "params.learning_rate",
        "metrics.final_score",
    ]
    if c in runs.columns
]

print(runs[columns].head(10))

                             run_id    status params.learning_rate  \
0  95cff2ee68e0485c8890b5e3c811ce51  FINISHED                 None   
1  95ae66b5be2d4441a64b16c317de1f66  FINISHED                 None   
2  a2809c84226f4744af38d732285a1ef2  FINISHED                 None   
3  f0fc64f7ca8f46eb95d4ab4463bfa35c  FINISHED                 None   
4  98fb4895c74843559783ebbe9351a8c7  FINISHED                 None   
5  6700e87c9ba64e5f8eece1950bb9e51b  FINISHED                 None   
6  89310a9f8c3749d59cd8ce7fb50d6571  FINISHED                 0.01   
7  ca6fd18a05394814a73cb6b577e6743b  FINISHED                 0.01   
8  03490738a3864907a7b1a33fe287a4ff  FINISHED                 0.01   
9  abc09bf35e984f9eac1fd3eaf516caa8  FINISHED                  0.1   

   metrics.final_score  
0                 0.95  
1                  NaN  
2                  NaN  
3                  NaN  
4                  NaN  
5                  NaN  
6                  NaN  
7                  NaN  
8   

## 13. Filtering runs

  filter expressions  parameters, metrics, tags .

:

```python
filter_string="metrics.rmse < 50"
```

:

```python
filter_string="params.model = 'random_forest'"
```

 filters       Runs.

          :

> Which runs satisfy my selection criteria?

In [18]:
# Cell 13 — filter runs by metric

filtered = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="metrics.rmse < 50",
    order_by=["metrics.rmse ASC"],
)

print("Matching runs:", len(filtered))
print(filtered.head(10))

Matching runs: 0
Empty DataFrame
Columns: [run_id, experiment_id, status, artifact_uri, start_time, end_time]
Index: []


## 14. Experiment-level metadata

 Run tags      .

  tags  Experiment :

```python
mlflow.set_experiment_tag(...)
mlflow.set_experiment_tags({...})
```

 Experiment Tags     Runs :

```text
project
team
business_owner
dataset_family
```

 Run Tags    .

In [19]:
# Cell 14 — experiment tags

mlflow.set_experiment(EXPERIMENT_NAME)

mlflow.set_experiment_tags({
    "project": "mlflow-learning",
    "team": "data-science",
    "owner": "student",
})

print("Experiment tags updated.")

Experiment tags updated.


## 15. Nested Runs — Useful for hyperparameter tuning

  :

```python
mlflow.start_run(nested=True)
```

:

```text
Parent Run
│
├── Trial 1
├── Trial 2
├── Trial 3
└── Trial 4
```

 Parent   tuning/search  child runs  trials.

:

```text
learning_rate × batch_size
```

 combination    child run.

  UI run organization .

In [20]:
# Cell 15 — parent run + child runs

learning_rates = [0.001, 0.01]
batch_sizes = [16, 32]

with mlflow.start_run(run_name="hyperparameter_search_parent"):
    mlflow.set_tag("run_type", "hyperparameter_search")

    for lr in learning_rates:
        for batch_size in batch_sizes:
            with mlflow.start_run(
                run_name=f"lr={lr}_batch={batch_size}",
                nested=True,
            ):
                score = 0.80 + (0.01 if lr == 0.01 else 0) - (0.01 if batch_size == 32 else 0)

                mlflow.log_params({
                    "learning_rate": lr,
                    "batch_size": batch_size,
                })
                mlflow.log_metric("score", score)

print("Nested hyperparameter runs created.")

Nested hyperparameter runs created.


## 16. `MlflowClient` — Low-level Low-Level API

 :

```python
client = MlflowClient()
```

 high-level API :

```python
mlflow.start_run()
mlflow.log_metric()
```

   training code.

 `MlflowClient`    entities :

- Experiments
- Runs
- Registered Models
- Model Versions
- Aliases
- Tags
- Descriptions

    CRUD  MLflow.

In [ ]:
# Cell 16 — inspect runs with MlflowClient

from mlflow import MlflowClient

client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=5,
)

for run in runs:
    print(
        run.info.run_id,
        run.info.status,
        run.data.metrics,
    )

## 17. Model Registry — Why do we need it?

Tracking answers:

> What happened during training?

Model Registry answers:

> Which model versions are managed as named models?

:

```text
Registered Model
    diabetes_predictor
        │
        ├── Version 1
        ├── Version 2
        └── Version 3
```

 Registry :

- versioning
- lineage
- aliases
- model/version metadata
- collaboration around model lifecycle

: **Registered Model   Run **.

Run  . Registry  model versions   .

In [ ]:
# Cell 17 — register a model automatically from a run

MODEL_NAME = "mlflow_master_diabetes_model"

with mlflow.start_run(run_name="registry_training") as run:
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
    )
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    signature = infer_signature(X_test, predictions)

    model_info = mlflow.sklearn.log_model(
        model,
        name="registry_model",
        signature=signature,
        registered_model_name=MODEL_NAME,
    )

    rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
    mlflow.log_metric("rmse", rmse)

    print("Run ID:", run.info.run_id)
    print("Logged model URI:", model_info.model_uri)
    print("Registered model:", MODEL_NAME)

## 18. Registered Model vs Model Version

 :

```text
MODEL_NAME = "mlflow_master_diabetes_model"
```

  **Registered Model**.

  model    MLflow  Version :

```text
mlflow_master_diabetes_model
    ├── v1
    ├── v2
    └── v3
```

 version    Run  .

   **lineage**:
  experiment/run   model

In [ ]:
# Cell 18 — inspect registered model versions

versions = list(
    client.search_model_versions(
        filter_string=f"name='{MODEL_NAME}'"
    )
)

versions = sorted(versions, key=lambda v: int(v.version))

for v in versions:
    print(
        f"version={v.version}, "
        f"run_id={v.run_id}, "
        f"aliases={v.aliases}, "
        f"tags={v.tags}"
    )

## 19. Model Version metadata

 :

- Description
- Tags

 Model Version.

:

```text
algorithm = random_forest
framework = sklearn
validation = passed
```

    model   metadata   governance   .

     lifecycle    .

In [ ]:
# Cell 19 — add model version description and tags

latest_version = versions[-1]

client.update_model_version(
    name=MODEL_NAME,
    version=latest_version.version,
    description="Random Forest trained on sklearn diabetes dataset.",
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=latest_version.version,
    key="algorithm",
    value="random_forest",
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=latest_version.version,
    key="validation",
    value="passed",
)

updated = client.get_model_version(
    MODEL_NAME,
    latest_version.version,
)

print("Version:", updated.version)
print("Description:", updated.description)
print("Tags:", updated.tags)

## 20. Modern Model Registry lifecycle — Aliases Instead of Stages

   :

```python
transition_model_version_stage(...)
```

:

```text
Staging
Production
Archived
```

   legacy.

   Model Stages deprecated.

:

```text
Registered Model
    │
    ├── v1
    ├── v2
    └── v3
         │
         └── alias: champion
```

 application :

```text
models:/<model-name>@champion
```

 :

   production model   application code.
   alias  version .

In [ ]:
# Cell 20 — create/update a modern "champion" alias

latest_version = versions[-1]

client.set_registered_model_alias(
    MODEL_NAME,
    "champion",
    latest_version.version,
)

champion = client.get_model_version_by_alias(
    MODEL_NAME,
    "champion",
)

print("Champion version:", champion.version)
print("Champion aliases:", champion.aliases)

## 21.  Alias   hard-coding a version

:

```python
models:/my_model/7
```

  application    version 7.

:

```python
models:/my_model@champion
```

 reference .

:

```text
champion → v7
```

:

```text
champion → v8
```

 application code  .

  deployment      configuration/code   model.

In [ ]:
# Cell 21 — load the model by alias

champion_uri = f"models:/{MODEL_NAME}@champion"

champion_model = mlflow.sklearn.load_model(champion_uri)

sample = X_test[:3]
predictions = champion_model.predict(sample)

print("Loaded:", champion_uri)
print("Sample shape:", sample.shape)
print("Predictions:", predictions)

## 22. Loading by exact version

  reproducibility .

:

```python
models:/my_model/3
```

   version .

 version URI  :

- reproducible testing
- debugging
- rollback analysis
- comparing exact model versions

 alias   reference   `champion`.

In [ ]:
# Cell 22 — load an exact model version

exact_version = versions[-1].version
version_uri = f"models:/{MODEL_NAME}/{exact_version}"

exact_model = mlflow.sklearn.load_model(version_uri)

print("Loaded:", version_uri)
print("Prediction:", exact_model.predict(X_test[:2]))

## 23. Loading from a Run URI

    model artifact   Run:

```text
runs:/<run_id>/<artifact_path>
```

    model artifact   training run    Registry.

  :

```text
runs:/...       → artifact produced by a specific run
models:/...     → managed model reference in Model Registry
```

 production lifecycle Registry + alias   .

In [ ]:
# Cell 23 — load a model directly from its producing run

# Find the run that produced the latest registered version.
source_run_id = latest_version.run_id
run_model_uri = f"runs:/{source_run_id}/registry_model"

run_model = mlflow.sklearn.load_model(run_model_uri)

print("Run model URI:", run_model_uri)
print("Prediction:", run_model.predict(X_test[:2]))

## 24. Downloading artifacts

MLflow   artifacts  local filesystem.

    workflow :

```text
remote artifact
    ↓
local directory
    ↓
inspect / process / load
```

     :

```python
mlflow.artifacts.download_artifacts(...)
```

  download    Registry     artifact  local path    .

In [ ]:
# Cell 24 — download a model artifact locally

download_dir = Path("downloaded_mlflow_model")

local_path = mlflow.artifacts.download_artifacts(
    artifact_uri=champion_uri,
    dst_path=str(download_dir),
)

print("Downloaded to:", local_path)

## 25. Backend Store vs Artifact Store

  architecture     :

```text
                 MLflow
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
   Backend Store         Artifact Store
          │                   │
          ▼                   ▼
 metadata                  files
 runs                      models
 params                    plots
 metrics                   JSON/CSV
 tags                      data files
```

### Backend Store
stores metadata.

### Artifact Store
  .

 docs   SQLite/PostgreSQL/MySQL/MSSQL  backend options S3/GCS/Azure Blob   artifact storage options.

In [ ]:
# Cell 25 — inspect the tracking configuration

print("Tracking URI:", mlflow.get_tracking_uri())

# The notebook uses SQLite as the local backend.
# Artifacts are managed by MLflow according to the experiment/run artifact location.
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment artifact location:")
print(experiment.artifact_location)

## 26. Local development architecture

 architecture:

```text
Python / Notebook
       │
       ├──────────────► SQLite backend
       │
       └──────────────► local artifact storage

                 ▲
                 │
             MLflow UI
```

   single-user development.

 supplied notebook        .

In [ ]:
# Cell 26 — print the local architecture configuration

print("LOCAL DEVELOPMENT")
print("-----------------")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)
print("Database:", DB_PATH.resolve())
print("UI command:")
print(f"mlflow server --backend-store-uri {TRACKING_URI} --host 127.0.0.1 --port 5000")

## 27. Tracking Server —   MLflow

 team environment      developer  SQLite .

:

```text
Developer A ─┐
Developer B ─┼──► MLflow Tracking Server
Developer C ─┘             │
                            ├──► PostgreSQL
                            └──► Artifact Store
```

Tracking Server  HTTP/REST endpoint UI.

 client :

```text
MLFLOW_TRACKING_URI=http://server:5000
```

  runs are stored in  .

In [ ]:
# Cell 27 — connect to a tracking server (example configuration)

# DO NOT run this cell unless a server is actually running.
# Replace the URL with your real MLflow Tracking Server.

TRACKING_SERVER_URI = "http://127.0.0.1:5000"

print("Example:")
print("mlflow.set_tracking_uri(TRACKING_SERVER_URI)")
print("mlflow.set_experiment('team_experiment')")

## 28. Team / production architecture

The architecture      MLOps :

```text
             Training Jobs
                  │
                  ▼
          MLflow Python SDK
                  │
                  ▼
         MLflow Tracking Server
             /            \
            /              \
           ▼                ▼
     PostgreSQL          S3 / MinIO
     metadata            artifacts
            \              /
             \            /
                  ▼
             Model Registry
                  │
             alias: champion
                  │
                  ▼
             FastAPI service
                  │
                  ▼
               /predict
```

 architecture   notebook    MLflow concepts.

In [ ]:
# Cell 28 — production-style configuration template

# This is a template only. Do not execute until the infrastructure exists.

PRODUCTION_TRACKING_URI = "http://mlflow-server:5000"

PRODUCTION_CONFIG = {
    "tracking_server": PRODUCTION_TRACKING_URI,
    "backend_store": "PostgreSQL",
    "artifact_store": "S3 or MinIO",
    "model_registry": "MLflow Model Registry",
    "production_reference": "models:/your_model@champion",
}

for key, value in PRODUCTION_CONFIG.items():
    print(f"{key}: {value}")

## 29. Debugging —   MLflow UI    Runs

      .

 :

```text
Python code
    ↓
writes to DB A

MLflow UI
    ↓
reads DB B
```

 UI .

    backend store       UI/server.

:

```python
mlflow.set_tracking_uri("sqlite:///mlflow_master.db")
```

    MLflow    .

In [ ]:
# Cell 29 — verify the database/URI before opening the UI

print("Current MLflow tracking URI:")
print(mlflow.get_tracking_uri())

print("\nExpected local UI/server backend:")
print(TRACKING_URI)

print("\nRun this in a terminal:")
print(f"mlflow server --backend-store-uri {TRACKING_URI} --host 127.0.0.1 --port 5000")

## 30. A complete end-to-end MLflow workflow

  workflow:

```text
1. Configure Tracking URI
          ↓
2. Select/Create Experiment
          ↓
3. Start Run
          ↓
4. Train Model
          ↓
5. Log Params
          ↓
6. Calculate Metrics
          ↓
7. Log Metrics
          ↓
8. Log Tags
          ↓
9. Log Artifacts
          ↓
10. Log Model + Signature
          ↓
11. Search/Compare Runs
          ↓
12. Register the selected model
          ↓
13. Create Model Version
          ↓
14. Add Version metadata
          ↓
15. Point an Alias such as "champion"
          ↓
16. Application loads models:/name@champion
```

If you understand this flow    MLflow Tracking + Registry.

In [ ]:
# Cell 30 — compact end-to-end template

import mlflow
import numpy as np
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

END_TO_END_MODEL = "mlflow_end_to_end_model"

with mlflow.start_run(run_name="end_to_end_training") as run:
    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
    signature = infer_signature(X_test, predictions)

    mlflow.log_params(model.get_params())
    mlflow.log_metric("rmse", rmse)
    mlflow.set_tags({
        "workflow": "end_to_end",
        "dataset": "sklearn_diabetes",
    })

    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        registered_model_name=END_TO_END_MODEL,
    )

print("Training run:", run.info.run_id)
print("Model:", END_TO_END_MODEL)
print("RMSE:", rmse)
print("Logged model:", model_info.model_uri)

# Point "champion" to the newest registered version.
client = MlflowClient()
model_versions = sorted(
    list(client.search_model_versions(
        filter_string=f"name='{END_TO_END_MODEL}'"
    )),
    key=lambda v: int(v.version),
)

newest = model_versions[-1]

client.set_registered_model_alias(
    END_TO_END_MODEL,
    "champion",
    newest.version,
)

print("Champion version:", newest.version)
print("Production-style URI:", f"models:/{END_TO_END_MODEL}@champion")

# 31. What each MLflow object means — final reference

| Object | Meaning | Example |
|---|---|---|
| Tracking URI |  MLflow  | `sqlite:///mlflow.db` |
| Experiment |  Runs | `diabetes_prediction` |
| Run |   | `run_id=abc123` |
| Parameter | configuration | `n_estimators=100` |
| Metric | performance | `rmse=42.1` |
| Tag | metadata | `environment=dev` |
| Artifact | file | `plot.png` |
| MLflow Model | model package | sklearn model |
| Model Signature | input/output schema | columns + dtypes |
| Registered Model |  model  | `diabetes_predictor` |
| Model Version |   registered model | `v7` |
| Alias | reference  | `champion → v7` |
| Backend Store | metadata storage | PostgreSQL/SQLite |
| Artifact Store | file storage | S3/MinIO/local |
| Tracking Server | HTTP service + UI | `http://...:5000` |

##

```text
Tracking:
"What happened?"

Registry:
"Which model version are we managing/deploying?"
```

       .

# 32. API Cheat Sheet

## Tracking

```python
mlflow.set_tracking_uri(uri)
mlflow.set_experiment(name)

mlflow.start_run()
mlflow.end_run()

mlflow.log_param(key, value)
mlflow.log_params(dictionary)

mlflow.log_metric(key, value)
mlflow.log_metrics(dictionary)

mlflow.set_tag(key, value)
mlflow.set_tags(dictionary)

mlflow.log_artifact(path)
mlflow.log_artifacts(directory)
mlflow.log_dict(dictionary, path)
mlflow.log_figure(figure, path)

mlflow.search_runs(...)
```

## Models

```python
mlflow.sklearn.log_model(...)
mlflow.sklearn.load_model(...)

from mlflow.models import infer_signature
signature = infer_signature(inputs, outputs)
```

## Registry / Client

```python
from mlflow import MlflowClient

client = MlflowClient()

client.search_model_versions(...)
client.get_model_version(...)

client.update_model_version(...)
client.set_model_version_tag(...)

client.set_registered_model_alias(...)
client.get_model_version_by_alias(...)
client.delete_registered_model_alias(...)
```

## Model URIs

```text
runs:/<run_id>/<artifact_path>

models:/<model_name>/<version>

models:/<model_name>@<alias>
```

# 33. Legacy vs Modern MLflow

##    supplied crash-course

```python
client.transition_model_version_stage(...)
```

:

```text
Staging
Production
Archived
```

## Modern approach

```python
client.set_registered_model_alias(
    "model_name",
    "champion",
    version,
)
```

:

```text
models:/model_name@champion
```

**Do not build a new project  Stages     Aliases.**

 supplied notebook       notebook  current MLflow documentation   registry lifecycle.

# 34. Study checklist

Do not consider yourself proficient MLflow until you can       :

- [ ]    Experiment Run
- [ ]    Parameter Metric Tag
- [ ]   Artifact
- [ ]    Backend Store Artifact Store
- [ ]   Tracking URI
- [ ]   `step`  metrics
- [ ]   Model Signature
- [ ]     Run
- [ ]   Nested Runs
- [ ]   `MlflowClient`
- [ ]    Logged Model Registered Model
- [ ]   Model Version
- [ ]   Lineage
- [ ]  Aliases   hard-coded production versions
- [ ]    `runs:/...` `models:/...`
- [ ]   Tracking Server  team environment
- [ ]    metadata artifacts
- [ ]    "UI is empty"

             code cell .

# Sources used

## Supplied source
`mlflow_crash_course.ipynb`

## Official MLflow documentation
- Tracking: https://mlflow.org/docs/latest/ml/tracking
- Architecture: https://mlflow.org/docs/latest/self-hosting/architecture/overview/
- Backend Stores: https://mlflow.org/docs/latest/self-hosting/architecture/backend-store/
- Artifact Stores: https://mlflow.org/docs/latest/self-hosting/architecture/artifact-store/
- Tracking Server: https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/
- Model Registry: https://mlflow.org/docs/latest/ml/model-registry/
- Model Registry Workflow: https://mlflow.org/docs/latest/ml/model-registry/workflow
- Model Signatures: https://mlflow.org/docs/latest/ml/model/signatures/
- Python API: https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html